In [1]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel

# Download NLTK Resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sean\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Sean\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Sean\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
documents = [
    "Rafael Nadal Joins Roger Federer in Missing U.S. Open",
    "Rafael Nadal Is Out of the Australian Open",
    "Biden Announces Virus Measures",
    "Biden's Virus Plans Meet Reality",
    "Where Biden's Virus Plan Stands"
]

In [3]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Tokenize the text into words and convert to lowercase
    tokens = word_tokenize(text.lower())
    # Filter out non-alphanumeric tokens
    tokens = [token for token in tokens if token.isalnum()]
    # Remove stopwords from the tokens
    tokens = [token for token in tokens if token not in stop_words]
    # Lemmatize each token
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]


In [4]:
# Create a Gensim Dictionary object
dictionary = corpora.Dictionary(preprocessed_documents)
# Convert preprocessed docs into a bag-of-words representation using the dictionary
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]


In [5]:
lda_model = LdaModel(corpus, num_topics=2, id2word=dictionary, passes=15)

In [7]:
import pandas as pd
# Empty list to store dominant topic labels
article_labels = []

# Iterate over each processed document
for i, doc in enumerate(preprocessed_documents):
    # Convert document to bag-of-words representation
    bow = dictionary.doc2bow(doc)
    # Get list of topic probabilities
    topics = lda_model.get_document_topics(bow)
    # Determine topic with highest probability
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    # Append to the list (Fixed the 'appenf' typo here)
    article_labels.append(dominant_topic)

# --- Create DataFrame and Print ---
df = pd.DataFrame({"Article": documents, "Topic": article_labels})

print("Table with Articles and Topic:")
print(df)

Table with Articles and Topic:
                                             Article  Topic
0  Rafael Nadal Joins Roger Federer in Missing U....      0
1         Rafael Nadal Is Out of the Australian Open      0
2                     Biden Announces Virus Measures      1
3                   Biden's Virus Plans Meet Reality      1
4                    Where Biden's Virus Plan Stands      1


In [8]:
# Print the top terms for each topic
print("Top Terms for Each Topic:")
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    # Split the string (e.g., '0.034*"biden" + 0.021*"virus"') into individual word parts
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        # Separate the weight (0.034) from the word ("biden")
        weight, word = term.split("*")
        # Print clearly: - "biden" (weight: 0.034)
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

Top Terms for Each Topic:
Topic 0:
- "rafael" (weight: 0.130)
- "open" (weight: 0.130)
- "nadal" (weight: 0.130)
- "federer" (weight: 0.078)
- "missing" (weight: 0.078)
- "roger" (weight: 0.078)
- "join" (weight: 0.078)
- "australian" (weight: 0.078)
- "meet" (weight: 0.029)
- "reality" (weight: 0.029)

Topic 1:
- "biden" (weight: 0.167)
- "virus" (weight: 0.167)
- "plan" (weight: 0.118)
- "announces" (weight: 0.072)
- "measure" (weight: 0.072)
- "stand" (weight: 0.072)
- "reality" (weight: 0.069)
- "meet" (weight: 0.069)
- "australian" (weight: 0.024)
- "nadal" (weight: 0.024)

